## Adult Dataset

In [1]:
# Vérification que jutilise le bon environnement que jai cree sur anconda
import sys
print(sys.executable)  # Affiche le chemin de Python

C:\Users\Bsmok\anaconda3\envs\cyber_rules\python.exe


In [3]:
# Test des imports pour s'assurer que tout est installé
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, accuracy_score, balanced_accuracy_score
from aix360.algorithms.rbm import FeatureBinarizer
import time
import warnings
import wittgenstein as lw
import explainer
import xgboost as xgb
warnings.filterwarnings("ignore")

print(f" NumPy: {np.__version__}")
print(f" Pandas: {pd.__version__}")
print(" Environnement cyber_rules fonctionne!")

 NumPy: 1.23.5
 Pandas: 1.5.3
 Environnement cyber_rules fonctionne!


In [4]:
# Définition des types de données
data_type = {'age': float,
             'workclass': str,
             'fnlwgt': float,
             'education': str,
             'education-num': float,
             'marital-status': str,
             'occupation': str,
             'relationship': str,
             'race': str,
             'sex': str,
             'capital-gain': float,
             'capital-loss': float,
             'native-country': str,
             'hours-per-week': float,
             'label': str}

# noms des colones
col_names = ['age', 'workclass', 'fnlwgt', 'education',
             'education-num', 'marital-status', 'occupation',
             'relationship', 'race', 'sex',
             'capital-gain', 'capital-loss', 'hours-per-week',
             'native-country', 'label']

# chargement des data
df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data',
                 header=None,
                 delimiter=', ',
                 engine='python',
                 names=col_names,
                 dtype=data_type)

In [5]:
# Renommage des colonnes (- en _)
df.columns = df.columns.str.replace('-', '_')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             32561 non-null  float64
 1   workclass       32561 non-null  object 
 2   fnlwgt          32561 non-null  float64
 3   education       32561 non-null  object 
 4   education_num   32561 non-null  float64
 5   marital_status  32561 non-null  object 
 6   occupation      32561 non-null  object 
 7   relationship    32561 non-null  object 
 8   race            32561 non-null  object 
 9   sex             32561 non-null  object 
 10  capital_gain    32561 non-null  float64
 11  capital_loss    32561 non-null  float64
 12  hours_per_week  32561 non-null  float64
 13  native_country  32561 non-null  object 
 14  label           32561 non-null  object 
dtypes: float64(6), object(9)
memory usage: 3.7+ MB


In [6]:
TARGET_COLUMN = 'label'
POS_VALUE = '>50K' # Setting positive value of the label for which we train
values_dist = df[TARGET_COLUMN].value_counts()

#ici on defini la separation des data de 20 et 80% 
train, test = train_test_split(df, test_size=0.2, random_state=42)
# Split the data set into 80% training and 20% test set
print('Training set:')
print(train[TARGET_COLUMN].value_counts())
print('Test set:')
print(test[TARGET_COLUMN].value_counts())

y_train = train[TARGET_COLUMN].apply(lambda x: 1 if x == POS_VALUE else 0)
x_train = train.drop(columns=[TARGET_COLUMN])

y_test = test[TARGET_COLUMN].apply(lambda x: 1 if x == POS_VALUE else 0)
x_test = test.drop(columns=[TARGET_COLUMN])

Training set:
<=50K    19778
>50K      6270
Name: label, dtype: int64
Test set:
<=50K    4942
>50K     1571
Name: label, dtype: int64


In [7]:
from sklearn.preprocessing import LabelEncoder

mapping = {}
# Label encode categorical columns
#Convertit les valeurs textuelles en nombres
le = LabelEncoder()
for col in x_train.select_dtypes(include=[object]).columns:
    x_train[col] = le.fit_transform(x_train[col])
    x_test[col] = le.transform(x_test[col])
    mapping[col] = dict(zip(le.classes_, le.transform(le.classes_)))

In [8]:
def calculate_coverage_and_correctness(X_test, y_test, mapping):
    """
    Calculates:
    - Coverage: % of test samples covered by the rule.
    - Correctness: % of covered instances that are actually '1' in y_test.

    :param X_test: DataFrame containing test samples.
    :param y_test: Ground truth labels.
    :param mapping: Dictionary mapping categorical values to encoded integers.
    :return: Coverage percentage and correctness percentage.
    """
    # Get encoded value for 'Married-civ-spouse'
    marital_status_encoded = mapping['marital_status'].get('Married-civ-spouse', -1)

    # Get encoded values for excluded occupations
    excluded_occupations = ['Craft-repair', 'Farming-fishing', 'Handlers-cleaners', 'Other-service']
    excluded_occupation_codes = [mapping['occupation'].get(occ, -1) for occ in excluded_occupations]

    # Apply rule conditions
    rule_mask = (
        (X_test['age'] > 26.0) &
        (X_test['education_num'] > 9.0) &
        (X_test['marital_status'] == marital_status_encoded) &
        (~X_test['occupation'].isin(excluded_occupation_codes))
    )

    # Compute Coverage
    covered_count = rule_mask.sum()
    total_count = len(X_test)
    coverage_pct = (covered_count / total_count) * 100 if total_count > 0 else 0

    # Extract corresponding labels
    covered_labels = y_test[rule_mask]

    # Compute Correctness (Percentage of covered instances where y_test == 1)
    correct_count = (covered_labels == 1).sum()
    correctness_pct = (correct_count / covered_count) * 100 if covered_count > 0 else 0

    # Print Results
    print(f"Covered Instances: {covered_count} / {total_count} ({coverage_pct:.2f}%)")
    print(f"Correct Instances: {correct_count} / {covered_count} ({correctness_pct:.2f}%)")

    return coverage_pct, correctness_pct

In [10]:
print('train')
calculate_coverage_and_correctness(x_train, y_train, mapping)
print('test')
calculate_coverage_and_correctness(x_test, y_test, mapping)

train
Covered Instances: 5308 / 26048 (20.38%)
Correct Instances: 3510 / 5308 (66.13%)
test
Covered Instances: 1292 / 6513 (19.84%)
Correct Instances: 849 / 1292 (65.71%)


(19.83724857976355, 65.71207430340557)

In [17]:
#ici on n’est plus sur une seule règle, mais sur un ensemble très large de règles combinées avec des OR
def calculate_coverage_and_correctness(X_test, y_test, mapping):
    """
    Calculates:
    - Coverage: % of test samples covered by the rule.
    - Correctness: % of covered instances that are actually '1' in y_test.

    :param X_test: DataFrame containing test samples.
    :param y_test: Ground truth labels.
    :param mapping: Dictionary mapping categorical values to encoded integers.
    :return: Coverage percentage and correctness percentage.
    """
    # Get encoded value for 'Married-civ-spouse'
    marital_status_encoded = mapping['marital_status'].get('Married-civ-spouse', -1)

    # Get encoded values for categorical conditions
    occupation_exec_managerial = mapping['occupation'].get('Exec-managerial', -1)
    occupation_prof_specialty = mapping['occupation'].get('Prof-specialty', -1)
    education_some_college = mapping['education'].get('Some-college', -1)
    relationship_wife = mapping['relationship'].get('Wife', -1)
    workclass_federal_gov = mapping['workclass'].get('Federal-gov', -1)
    workclass_private = mapping['workclass'].get('Private', -1)
    workclass_self_emp_inc = mapping['workclass'].get('Self-emp-inc', -1)

    # Apply rule conditions (logical OR for multiple conditions)
    rule_mask = (
    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['age'] >= 37.0) & 
     (X_test['education_num'] <= 11.0) & (X_test['hours_per_week'] >= 38.0) & 
     (X_test['workclass'] == mapping['workclass']['Federal-gov'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 14.0) & (X_test['capital_loss'] >= 1741.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['education_num'] <= 11.0) & 
     (X_test['hours_per_week'] >= 38.0) & (X_test['fnlwgt'] >= 139000.0) & 
     (X_test['fnlwgt'] <= 145098.0) & (X_test['age'] <= 54.0) & (X_test['age'] >= 51.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['education_num'] <= 10.0) &
     (X_test['hours_per_week'] >= 38.0) & (X_test['age'] <= 59.0) & 
     (X_test['hours_per_week'] <= 46.0) & (X_test['age'] >= 58.0) & 
     (X_test['occupation'] == mapping['occupation']['Adm-clerical'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['education_num'] <= 11.0) & 
     (X_test['fnlwgt'] >= 140664.0) & (X_test['hours_per_week'] >= 44.0) & 
     (X_test['hours_per_week'] <= 52.0) & (X_test['age'] >= 57.0) & 
     (X_test['fnlwgt'] <= 155256.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['education_num'] <= 10.0) & 
     (X_test['hours_per_week'] >= 40.0) & (X_test['hours_per_week'] <= 48.0) & 
     (X_test['education'] == mapping['education']['Some-college']) & 
     (X_test['age'] >= 48.0) & (X_test['fnlwgt'] >= 281540.0) & (X_test['age'] <= 52.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['occupation'] == mapping['occupation']['Prof-specialty']) & 
     (X_test['education_num'] >= 14.0) & (X_test['education'] == mapping['education']['Prof-school'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['hours_per_week'] >= 36.0) & (X_test['occupation'] == mapping['occupation']['Prof-specialty']) & 
     (X_test['education_num'] >= 14.0) & (X_test['hours_per_week'] <= 40.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 13.0) & (X_test['occupation'] == mapping['occupation']['Exec-managerial']) & 
     (X_test['age'] >= 45.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 13.0) & (X_test['hours_per_week'] >= 31.0) & 
     (X_test['capital_loss'] >= 1741.0) & (X_test['capital_loss'] <= 1977.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 10.0) & (X_test['age'] >= 36.0) & 
     (X_test['capital_gain'] >= 5060.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 13.0) & (X_test['hours_per_week'] >= 35.0) & 
     (X_test['age'] >= 29.0) & (X_test['occupation'] == mapping['occupation']['Exec-managerial'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 12.0) & (X_test['hours_per_week'] >= 41.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 11.0) & (X_test['age'] >= 48.0) & 
     (X_test['hours_per_week'] <= 40.0) & (X_test['age'] <= 61.0)) |

     ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['age'] >= 34.0) & (X_test['education_num'] >= 11.0) & 
     (X_test['occupation'] == mapping['occupation']['Exec-managerial'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 10.0) & (X_test['age'] >= 36.0) & 
     (X_test['capital_loss'] >= 1848.0) & (X_test['capital_loss'] <= 1977.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['age'] >= 36.0) & 
     (X_test['education_num'] <= 11.0) & (X_test['fnlwgt'] >= 117073.0) & 
     (X_test['occupation'] == mapping['occupation']['Tech-support'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 12.0) & (X_test['fnlwgt'] <= 306982.0) & 
     (X_test['fnlwgt'] >= 191364.0) & (X_test['hours_per_week'] >= 38.0) & 
     (X_test['age'] >= 44.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 10.0) & (X_test['age'] >= 31.0) & 
     (X_test['occupation'] == mapping['occupation']['Exec-managerial']) & 
     (X_test['workclass'] == mapping['workclass']['Self-emp-inc']) & 
     (X_test['fnlwgt'] >= 199352.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['education_num'] <= 10.0) & 
     (X_test['age'] >= 47.0) & (X_test['workclass'] == mapping['workclass']['Private']) & 
     (X_test['fnlwgt'] <= 145574.0) & (X_test['age'] <= 50.0) & 
     (X_test['fnlwgt'] >= 102821.0) & (X_test['hours_per_week'] >= 41.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 10.0) & (X_test['hours_per_week'] >= 44.0) & 
     (X_test['fnlwgt'] >= 218521.0) & (X_test['fnlwgt'] <= 255667.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['age'] >= 34.0) & (X_test['education_num'] >= 12.0) & 
     (X_test['occupation'] == mapping['occupation']['Prof-specialty']) & 
     (X_test['fnlwgt'] >= 193769.0) & (X_test['hours_per_week'] >= 39.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['capital_gain'] >= 5060.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['occupation'] == mapping['occupation']['Exec-managerial']) & 
     (X_test['workclass'] == mapping['workclass']['Private']) & 
     (X_test['age'] <= 56.0) & (X_test['age'] >= 40.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['education_num'] <= 11.0) & 
     (X_test['age'] >= 45.0) & (X_test['workclass'] == mapping['workclass']['Self-emp-inc']) & 
     (X_test['fnlwgt'] <= 194995.0) & (X_test['hours_per_week'] >= 46.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 10.0) & (X_test['age'] >= 37.0) & 
     (X_test['occupation'] == mapping['occupation']['Sales']) & 
     (X_test['age'] <= 48.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 10.0) & (X_test['age'] >= 28.0) & 
     (X_test['occupation'] == mapping['occupation']['Prof-specialty']) & 
     (X_test['relationship'] == mapping['relationship']['Wife']) & 
     (X_test['hours_per_week'] <= 35.0) & (X_test['age'] <= 42.0) & 
     (X_test['capital_gain'] <= 0.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['age'] >= 34.0) & 
     (X_test['education_num'] <= 11.0) & (X_test['hours_per_week'] >= 45.0) & 
     (X_test['fnlwgt'] >= 260578.0) & 
     (X_test['education'] == mapping['education']['Some-college'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['age'] >= 36.0) & 
     (X_test['education_num'] <= 11.0) & (X_test['hours_per_week'] >= 38.0) & 
     (X_test['capital_loss'] >= 1741.0) & (X_test['capital_loss'] <= 1977.0))
)
    # Compute Coverage
    covered_count = rule_mask.sum()
    total_count = len(X_test)   
    coverage_pct = (covered_count / total_count) * 100 if total_count > 0 else 0
    # Extract corresponding labels
    covered_labels = y_test[rule_mask]

    # Compute Correctness (Percentage of covered instances where y_test == 1)
    correct_count = (covered_labels == 1).sum()
    correctness_pct = (correct_count / covered_count) * 100 if covered_count > 0 else 0

    # Print Results
    print(f"Covered Instances: {covered_count} / {total_count} ({coverage_pct:.2f}%)")
    print(f"Correct Instances: {correct_count} / {covered_count} ({correctness_pct:.2f}%)")

    return coverage_pct, correctness_pct

In [19]:
print('train')
calculate_coverage_and_correctness(x_train, y_train, mapping)
print('test')
calculate_coverage_and_correctness(x_test, y_test, mapping)

train
Covered Instances: 4569 / 26048 (17.54%)
Correct Instances: 3504 / 4569 (76.69%)
test
Covered Instances: 1130 / 6513 (17.35%)
Correct Instances: 843 / 1130 (74.60%)


(17.349915553508367, 74.60176991150442)

On évalue un ensemble de regles combinées par des OU/OR logiques. Une instance est couverte si elle satisfait au moins une règle. La couverture est d’environ 17%, et la précision des règles est d’environ 75%. Comparé à la regle précédente, on a réduit la couverture mais augmenté la fiabilité, ce qui montre que les regles sont plus sélectives mais plus robustes.

In [22]:
# train xgboost

model = xgb.XGBClassifier()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy of XGBoost:', accuracy)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
print('Balanced accuracy of XGBoost:', balanced_accuracy)
recall = recall_score(y_test, y_pred)
print('Recall of XGBoost:', recall)
precision = precision_score(y_test, y_pred)
print('Precision of XGBoost:', precision)
from sklearn.metrics import f1_score
f1 = f1_score(y_test, y_pred)
print('F1 score of XGBoost:', f1)


Accuracy of XGBoost: 0.8751727314601566
Balanced accuracy of XGBoost: 0.808764159475891
Recall of XGBoost: 0.6804583068109484
Precision of XGBoost: 0.7746376811594203
F1 score of XGBoost: 0.7245001694340901


XGBoost obtient une accuracy de 87% et un F1-score de 72%. Il est plus performant globalement car il couvre toutes les données. Cependant, les règles extraites ont une précision comparable (~75%) tout en étant totalement interprétables. Cela montre le compromis entre performance et explicabilité.

In [25]:
#prediction sur le train
y_pred_train = model.predict(x_train)
#creation cmc
train_cmc = explainer.make_cmc(x_train, y_train, y_pred_train)
len_rows = train_cmc.shape[0]
#generation des regles exclusive
exclusive_rules, uids_ex = explainer.compute_rule_coverage(explainer.get_exclusive_rules(train_cmc, ['label', 'predicted', 'CMC']), len_rows)


100%|██████████| 14/14 [00:00<00:00, 14.69it/s]

Total Support: 25.26% (6580)
After removing duplicates:
Unique Coverage: 23.54% (6133)


Les regles exclusives couvrent environ 24% des données d’entraînement de manière totalement fidèle au modele. 
Cela signifie que 1/4 des décisions de XGBoost peut être expliqué par des regles simples et déterministes. 
Les 76% restants correspondent à des interactions plus complexes capturées par le modele mais difficiles a exprimer sous forme de regles simples.

In [28]:
import importlib
importlib.reload(explainer)

<module 'explainer' from 'C:\\Users\\Bsmok\\Desktop\\uqac\\Hiver_2026\\Projet_In_Github\\Confusion-Matrix-Rules-CMR-\\explainer.py'>

In [30]:
x_test['label'] = y_test
explainer.apply_exclusive_rules(x_test, exclusive_rules)

100%|██████████| 779/779 [00:00<00:00, 2308.70it/s]

Test set coverage: 19.67%
Test coverage F1-score: 0.97
--------------------------------------
Total Skipped: 5232 (80.33%)
 - Due to Conflicts: 0 (0.0%)
 - Due to No Rule Coverage: 5232 (80.33%)
--------------------------------------
Using Majority class to predict non-covered cases.
F1-score (if using entire dataset): 0.7829


Sur le dat, les regles exclusives couvrent environ 20% des données et atteignent un F1-score de 0.97 sur ces instances, ce qui est une bonne généralisation.
 Mais, elles ne couvrent pas les cas plus complexes. En combinant les regles avec une prédiction par classe majoritaire pour les non-couverts, on obtient un F1 global de 0.78, proche du modèle complet. 
 --> les regles capturent une partie robuste et stable du comportement du modele.

In [33]:
x_test['label'] = y_test
explainer.apply_exclusive_rules(x_test, exclusive_rules, model)

100%|██████████| 779/779 [00:00<00:00, 1940.97it/s]

Test set coverage: 19.67%
Test coverage F1-score: 0.97
--------------------------------------
Total Skipped: 5232 (80.33%)
 - Due to Conflicts: 0 (0.0%)
 - Due to No Rule Coverage: 5232 (80.33%)
--------------------------------------
Using model to predict non-covered cases.
F1-score (if using entire dataset): 0.8699


En utilisant une approche hybride, les regles exclusives expliquent environ 20% des décisions avec un F1-score de 0.97, tandis que le modele XGBoost prend en charge les cas restants. 
    Cette combinaison atteint un F1 global de 0.87, proche du modèle complet, tout en apportant une couche significative d’explicabilité.

2.	Faire fonctionner le code de brcg et ripper, soit via une librairie ou en recréant le code directement à partir des algorithmes. 

Pour cette question, j’ai choisi l’option : "via une librairie existante"

BRCG	
    libr : aix360	
        Librairie IBM AI Explainability 360. Contient BooleanRuleCG pour entraîner directement un modèle basé sur regles à partir de variables binaires.
RIPPER	
    libra : wittgenstein	
    Librairie spécialisée en rule learning, fournit RIPPER() pour l’apprentissage de regles interprétables avec pruning et controle du nombre de regles.

In [37]:
#BRCG (Boolean Rule Column Generation)
from aix360.algorithms.rbm import FeatureBinarizer, BooleanRuleCG
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# BRCG fonctionne uniquement avec des variables binaires (0/1)
# On doit donc transformer les données en format booléen
fb = FeatureBinarizer(negations=True)

x_train_bin = fb.fit_transform(x_train)
x_test_bin = fb.transform(x_test)

# Création du modèle BRCG
# lambda0 et lambda1 contrôlent la complexité des règles
start_time = time.time()

brcg = BooleanRuleCG(
    lambda0=0.001,
    lambda1=0.001,
   # solver='ECOS',
    verbose=False
)

# Entraînement du modèle sur les données d’apprentissage
brcg.fit(x_train_bin, y_train)

train_time = time.time() - start_time

# Prédiction sur les données de test
y_pred_brcg = brcg.predict(x_test_bin)

# Calcul des métriques de performance
test_acc_brcg = accuracy_score(y_test, y_pred_brcg)
test_f1_brcg = f1_score(y_test, y_pred_brcg, average='weighted')
test_precision_brcg = precision_score(y_test, y_pred_brcg, average='weighted', zero_division=0)
test_recall_brcg = recall_score(y_test, y_pred_brcg, average='weighted', zero_division=0)

print("Résultats BRCG")
print("Accuracy:", test_acc_brcg)
print("F1-score:", test_f1_brcg)
print("Precision:", test_precision_brcg)
print("Recall:", test_recall_brcg)

# Affichage des règles générées par le modèle
try:
    print("Règles générées :")
    print(brcg.explain())
except:
    print("Impossible d'afficher les règles")

Skipping column 'label': data type cannot be handled
Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved
Iteration: 1, Objective: 0.2181
Iteration: 2, Objective: 0.1897
Iteration: 3, Objective: 0.1843
Iteration: 4, Objective: 0.1843
Iteration: 5, Objective: 0.1843
Résultats BRCG
Accuracy: 0.8189774297558728
F1-score: 0.8026137894465102
Precision: 0.8063169426792735
Recall: 0.8189774297558728
Règles générées :
{'isCNF': False, 'rules': ['education_num > 11.00 AND relationship <= 0.00', 'education_num > 11.00 AND relationship > 4.00']}


Ici les règles sont simples, compactes et Très interprétables
Peu nombreuses → modèle concis

Notre seconde library : RIPPER qui fonctionne différemment :

Apprentissage séquentiel de règles
Ajout et pruning de règles (simplifier les regles générées)
Optimisation selon la description length

In [40]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# RIPPER nécessite que la variable cible soit incluse dans le DataFrame
train_data = x_train.copy()
train_data['label'] = y_train.values

test_data = x_test.copy()
test_data['label'] = y_test.values

# Création du modèle RIPPER
# prune_size : proportion des données utilisées pour simplifier les règles
# k : nombre d'itérations d’apprentissage
start_time = time.time()

ripper = lw.RIPPER(
    prune_size=0.33,
    k=2,
   # dl_allow_crease=True,
    max_rules=50
)

# Entraînement du modèle
ripper.fit(
    train_data,
    class_feat='label',
    pos_class=1,
    neg_class=0
)

train_time = time.time() - start_time

# Prédictions sur les données de test
y_pred_ripper = ripper.predict(test_data)

# Calcul des métriques
test_acc_ripper = accuracy_score(y_test, y_pred_ripper)
test_f1_ripper = f1_score(y_test, y_pred_ripper, average='weighted')
test_precision_ripper = precision_score(y_test, y_pred_ripper, average='weighted', zero_division=0)
test_recall_ripper = recall_score(y_test, y_pred_ripper, average='weighted', zero_division=0)

print("Résultats RIPPER")
print("Accuracy:", test_acc_ripper)
print("F1-score:", test_f1_ripper)
print("Precision:", test_precision_ripper)
print("Recall:", test_recall_ripper)

# Affichage des règles apprises
print("Règles générées :")
print(ripper.ruleset_)

Résultats RIPPER
Accuracy: 0.8226623675725472
F1-score: 0.8094117082253981
Precision: 0.8107774834878473
Recall: 0.8226623675725472
Règles générées :
[[marital_status=2^education_num=>13.0^occupation=8.0-10.0^native_country=36.0-39.0^capital_loss=1742.4-2178.0] V [marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0^hours_per_week=48.0-55.0^fnlwgt=106712.6-131230.9] V [marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0^workclass=4^hours_per_week=48.0-55.0^age=37.0-41.0] V [marital_status=2^education_num=>13.0^occupation=8.0-10.0^native_country=36.0-39.0^capital_gain=9999.9-19999.8] V [marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0^workclass=4^capital_gain=9999.9-19999.8] V [marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0^capital_loss=1742.4-2178.0] V [marital_status=2^education_num=>13.0^occupation=3.0-4.0^hours_per_week=>55.0] V [marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0^workclass=4^hours_per_week=48.0-55.0^age=33.0-37.0] V [m

Analyse : 
    Plus nombreuses et détaillées que BRCG
    Toujours interprétables
    Permet de capturer des patterns plus complexes

In [42]:
 print("Comparaison des modèles")

print("XGBoost Accuracy:", accuracy)
print("XGBoost F1-score:", f1)

print("BRCG Accuracy:", test_acc_brcg)
print("BRCG F1-score:", test_f1_brcg)

print("RIPPER Accuracy:", test_acc_ripper)
print("RIPPER F1-score:", test_f1_ripper)

Comparaison des modèles
XGBoost Accuracy: 0.8751727314601566
XGBoost F1-score: 0.7245001694340901
BRCG Accuracy: 0.8189774297558728
BRCG F1-score: 0.8026137894465102
RIPPER Accuracy: 0.8226623675725472
RIPPER F1-score: 0.8094117082253981


Resumé : 
    Les modeles à règles (BRCG, RIPPER) sont légérement moins performants que XGBoost, mais ils sont totalement interprétables
    RIPPER -> performance légérement meilleure mais avec plus de regles
    BRCG -> modele plus compact avec des regles simple

Comparés à XGBoost(boite noire plus legerement plus performant), ces modèles montrent le compromis classique interprétabilité vs performance.

In [49]:
from sklearn.metrics import accuracy_score, f1_score
from aix360.algorithms.rbm import FeatureBinarizer, BooleanRuleCG
from tqdm import tqdm
import gc
     


In [51]:
# Fonction test BRCG + RIPPER

def test_models(X_train, y_train, X_test, y_test, brcg_sample=None):

    results = {}

    #  BRCG 
    try:
        fb = FeatureBinarizer(negations=True)
        
        # Si sampling demandé
        if brcg_sample is not None and len(X_train) > brcg_sample:
            idx = np.random.choice(len(X_train), brcg_sample, replace=False)
            X_train_samp = X_train.iloc[idx] if isinstance(X_train, pd.DataFrame) else X_train[idx]
            y_train_samp = y_train.iloc[idx] if isinstance(y_train, pd.Series) else y_train[idx]
        else:
            X_train_samp = X_train
            y_train_samp = y_train
        
        X_train_bin = fb.fit_transform(X_train_samp)
        X_test_bin = fb.transform(X_test)

        model = BooleanRuleCG(lambda0=0.001, lambda1=0.001, solver="SCS")
        model.fit(X_train_bin, y_train_samp)

        y_pred = model.predict(X_test_bin)

        results["brcg_acc"] = accuracy_score(y_test, y_pred)
        results["brcg_f1"] = f1_score(y_test, y_pred, average="weighted")

    except Exception as e:
        print("Erreur BRCG:", e)
        results["brcg_acc"] = None
        results["brcg_f1"] = None


    #  RIPPER
    try:
        train_df = X_train.copy()
        train_df["label"] = y_train

        ripper = lw.RIPPER()
        ripper.fit(train_df, class_feat="label")

        y_pred = ripper.predict(X_test)

        results["ripper_acc"] = accuracy_score(y_test, y_pred)
        results["ripper_f1"] = f1_score(y_test, y_pred, average="weighted")

    except Exception as e:
        print("Erreur RIPPER:", e)
        results["ripper_acc"] = None
        results["ripper_f1"] = None

    return results



# Les chemins des datasets


datasets = {
    "KDD99": "../Datasets/KDD99/",
    "BIG15": "../Datasets/BIG15/",
    "DoH20": "../Datasets/DoH20/",
    "UNSW-NB15": "../Datasets/UNSW-NB15/processed/",
    "CIC2017": "../Datasets/CIC2017/"
}


results_all = {}


# Boucle principale

for name, path in tqdm(datasets.items(), desc="Testing datasets"):

    try:

     # definir variable  brcg_sample
        brcg_sample= None
        # CIC2017
        if name == "CIC2017":
            X_train = pd.read_parquet(path + "X_train.parquet")
            X_test = pd.read_parquet(path + "X_test.parquet")
            y_train = pd.read_csv(path + "y_train.csv").squeeze()
            y_test = pd.read_csv(path + "y_test.csv").squeeze()
            #ajout de mon sampling
            brcg_sample= 100000
            


        #  UNSW 
        elif name == "UNSW-NB15":

            train = pd.read_csv(path + "le_train.csv")
            test = pd.read_csv(path + "le_test.csv")

            X_train = train.drop(["id", "attack_cat", "label"], axis=1)
            y_train = train["label"].values

            X_test = test.drop(["id", "attack_cat", "label"], axis=1)
            y_test = test["label"].values


        # pour le reste   
        else:

            X_train = pd.read_csv(path + "X_train.csv")
            X_test = pd.read_csv(path + "X_test.csv")

            y_train = pd.read_csv(path + "y_train.csv").values.ravel()
            y_test = pd.read_csv(path + "y_test.csv").values.ravel()

    
        results_all[name] = test_models(X_train, y_train, X_test, y_test, brcg_sample)



        # Liberation de la memoire pour eviter le crash 

        del X_train, X_test, y_train, y_test

        if 'train' in locals():
            del train
        if 'test' in locals():
            del test

        gc.collect()

    except Exception as e:
        print(f"Erreur dataset {name}: {e}")
        results_all[name] = None
        gc.collect()



# Affichage résultats

print("\nRESULTATS\n")
print(f"{'Dataset':<12} {'BRCG Acc':<10} {'BRCG F1':<10} {'RIPPER Acc':<12} {'RIPPER F1'}")
print("-"*60)

for ds, res in results_all.items():
    if res and res["brcg_acc"] is not None:
        print(f"{ds:<12} "
              f"{res['brcg_acc']:.4f}   "
              f"{res['brcg_f1']:.4f}   "
              f"{res['ripper_acc']:.4f}   "
              f"{res['ripper_f1']:.4f}")

Testing datasets:   0%|          | 0/5 [00:00<?, ?it/s]

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved


Testing datasets:  20%|██        | 1/5 [1:09:35<4:38:20, 4175.04s/it]

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved


Testing datasets:  40%|████      | 2/5 [1:36:01<2:12:36, 2652.10s/it]

...

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved


Testing datasets:  80%|████████  | 4/5 [4:05:55<1:09:22, 4162.12s/it]

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved


Testing datasets: 100%|██████████| 5/5 [6:50:48<00:00, 4929.68s/it]  


RESULTATS

Dataset      BRCG Acc   BRCG F1    RIPPER Acc   RIPPER F1
------------------------------------------------------------
KDD99        0.9737   0.9736   0.9986   0.9986
BIG15        0.9820   0.9820   0.9909   0.9909
DoH20        0.9658   0.9616   0.9961   0.9961
UNSW-NB15    0.7728   0.7676   0.8334   0.8390
CIC2017      0.9232   0.9150   0.9971   0.9971


In [53]:
# Fonction test pour UGR16
def test_models_ugr16():
    results = {}
    path = "../Datasets/UGR16/"

    try:
        # Chargement UGR16
        X_train = pd.read_csv(path + "UGR16v1.Xtrain.csv")
        X_test = pd.read_csv(path + "UGR16v1.Xtest.csv")

        y_train_full = pd.read_csv(path + "UGR16v1.Ytrain.csv")
        y_test_full = pd.read_csv(path + "UGR16v1.Ytest.csv")

        # prends juste la colonne labelblacklist
        y_train = y_train_full["labelblacklist"]
        y_test = y_test_full["labelblacklist"]

        # Sécurité longueur
        y_train = y_train.iloc[:len(X_train)].reset_index(drop=True)
        y_test = y_test.iloc[:len(X_test)].reset_index(drop=True)

        # BRCG
        try:
            fb = FeatureBinarizer(negations=True)

            X_train_bin = fb.fit_transform(X_train)
            X_test_bin = fb.transform(X_test)

            model = BooleanRuleCG(lambda0=0.001, lambda1=0.001, solver="SCS")
            model.fit(X_train_bin, y_train)

            y_pred = model.predict(X_test_bin)

            results["brcg_acc"] = accuracy_score(y_test, y_pred)
            results["brcg_f1"] = f1_score(y_test, y_pred, average="weighted")

        except Exception as e:
            print("Erreur BRCG:", e)
            results["brcg_acc"] = None
            results["brcg_f1"] = None

        # RIPPER
        try:
            train_df = X_train.copy()
            train_df["label"] = y_train

            ripper = lw.RIPPER()
            ripper.fit(train_df, class_feat="label")

            y_pred = ripper.predict(X_test)

            results["ripper_acc"] = accuracy_score(y_test, y_pred)
            results["ripper_f1"] = f1_score(y_test, y_pred, average="weighted")

        except Exception as e:
            print("Erreur RIPPER:", e)
            results["ripper_acc"] = None
            results["ripper_f1"] = None

    except Exception as e:
        print("Erreur chargement UGR16:", e)
        return None

    return results


# Exécution
results_ugr16 = test_models_ugr16()

print("\nRESULTATS UGR16\n")
print(f"{'BRCG Acc':<12} {'BRCG F1':<12} {'RIPPER Acc':<12} {'RIPPER F1'}")
print("-"*55)

if results_ugr16:
    print(f"{results_ugr16['brcg_acc']}   "
          f"{results_ugr16['brcg_f1']}   "
          f"{results_ugr16['ripper_acc']}   "
          f"{results_ugr16['ripper_f1']}")

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Erreur BRCG: Invalid dimensions (0, 1741).
Erreur RIPPER: Couldn't infer name of positive target class from class feature: label. Try using parameter pos_class to specify which class label should be treated as positive, or try renaming your classes as booleans or 0, 1.

RESULTATS UGR16

BRCG Acc     BRCG F1      RIPPER Acc   RIPPER F1
-------------------------------------------------------
None   None   None   None


Interprétation des résultats

Les expérimentations montrent que RIPPER surpasse systématiquement BRCG sur l’ensemble des jeux de données testés. Sur KDD99, BIG15, DoH20 et CIC2017, RIPPER obtient des scores F1 très élevés, proches de 1, indiquant une excellente capacité à capturer les structures présentes dans les données.

BRCG reste performant, avec des scores F1 supérieurs à 0.91 sur la majorité des datasets, mais ses performances sont généralement inférieures à celles de RIPPER. La différence est particulièrement marquée sur CIC2017 et KDD99. De plus, BRCG nécessite un temps de calcul important, l’ensemble des expérimentations ayant duré près de 7 heures, ce qui confirme son coût computationnel élevé lié à la résolution d’un problème d’optimisation.

Le dataset UNSW-NB15 apparaît comme le plus difficile, avec des performances plus faibles pour les deux algorithmes, suggérant une structure plus complexe ou un niveau de bruit plus élevé.

Concernant UGR16, les deux algorithmes n’ont pas pu être exécutés correctement. BRCG a généré une erreur liée à des dimensions invalides, indiquant l’absence d’instances exploitables après prétraitement. RIPPER n’a pas pu identifier de classe positive, ce qui suggère un problème dans la structure des labels. Ce dataset ne pouvait donc pas être utilisé dans sa configuration actuelle.

Globalement, ces résultats montrent que les méthodes basées sur des règles peuvent atteindre des performances très élevées en détection d’intrusion, avec un avantage notable pour RIPPER en termes de précision et de robustesse.